In [25]:
import pandas as pd
from geopy.geocoders import Nominatim
import time

In [ ]:
import os

# variável de ambiente para construir o caminho
caminho_base = os.environ.get("CAMINHO_ARQUIVOS_CSV")
caminho_completo = os.path.join(caminho_base, "dados_tratados_v2", "dc_DadosPrimarios_tratado.csv")

df_colaboradores = pd.read_csv(caminho_completo)

In [ ]:
#df_colaboradores = pd.read_csv('../ArquivosExternos/dados_tratados_v2/dc_DadosPrimarios_tratado.csv')
df_colaboradores.head(10)

,ID,Nome,Email,CPF,Data de Nascimento,Telefone,Estado,Cidade,Endereço,Telefone_status,Situacao_CPF
0,7,Vanessa Nogueira,vanessa.nogueira@exemplo.com,345.123.678-90,1992-09-05,+55 (21) 9876-1234,RJ,Niterói,"Rua Mato Grosso, 192, Niterói - RJ",Inválido: celular com tamanho incorreto,Válido
1,8,Eduardo Araújo,edu.araújo@empresa.com,024.878.550-22,1980-07-30,+55 (11) 93220-4567,SP,São Paulo,"Rua Rio Branco, 831, São Paulo - SP",Válido,Válido
2,9,Mariana Silva,mariana@dominio.com.br,123.456.789-00,1999-06-15,+55 (19) 99765-3332,SP,Santos,"Rua Horizonte Azul, 779, Santos - SP",Válido,Válido
3,10,Rafael Costa,rafael.costa@email.com,789.654.321-00,1987-03-22,+55 (71) 91234-5678,BA,Feira de Santana,"Rua Cruzeiro do Sul, 162, Feira de Santana - BA",Válido,Válido
4,11,Thiago Moura,thiago@dominio.com,123.456.789-00,1990-01-12,+55 (21) 99999-0001,RJ,Niterói,"Rua do Sol, 121, Niterói - RJ",Válido,Válido
5,12,Camila Andrade,camila.andrade@dominio.com,987.654.321-00,1993-03-01,+55 (11) 91234-5678,SP,Santos,"Rua Esperança, 999, Santos - SP",Válido,Válido
6,13,Bruno Oliveira,bruno@empresa.com,000.000.000-00,1985-07-30,+55 (31) 9999-8888,MG,Contagem,"Rua Bela Vista, 648, Contagem - MG",Inválido: celular com tamanho incorreto,Inválido: todos os dígitos são iguais
7,14,Ana Carolina,ana.carol@email.com,321.654.987-00,1994-02-14,+55 (71) 98877-6655,BA,Feira de Santana,"Rua Acre, 555, Feira de Santana - BA",Válido,Válido
8,15,Fernando Reis,fernando.reis@empresa.com.br,789.654.123-00,1987-09-05,+55 (85) 92345-6789,CE,Juazeiro do Norte,"Rua Bela Vista, 658, Juazeiro do Norte - CE",Válido,Válido
9,16,Isabela Monteiro,isabela.monteiro@gmail.com,741.852.963-00,1982-10-31,+55 (98) 99876-5432,MA,São Luís,"Rua Dom Pedro I, 997, São Luís - MA",Válido,Válido


In [27]:
# Criar lista única de cidades
df_cidades = df_colaboradores[['Cidade', 'Estado']].drop_duplicates().reset_index(drop=True)
df_cidades.head(10)

,Cidade,Estado
0,Niterói,RJ
1,São Paulo,SP
2,Santos,SP
3,Feira de Santana,BA
4,Contagem,MG
5,Juazeiro do Norte,CE
6,São Luís,MA
7,Rio de Janeiro,RJ
8,Florianópolis,SC
9,Caxias do Sul,RS


In [ ]:
# Criar ID para cada cidade (começa no 1)
df_cidades['idCidade'] = df_cidades.index + 1
df_cidades['idCidade'] = df_cidades['idCidade'].astype(int)

print(df_cidades)

               Cidade Estado  idCidade
0             Niterói     RJ         1
1           São Paulo     SP         2
2              Santos     SP         3
3    Feira de Santana     BA         4
4            Contagem     MG         5
5   Juazeiro do Norte     CE         6
6            São Luís     MA         7
7      Rio de Janeiro     RJ         8
8       Florianópolis     SC         9
9       Caxias do Sul     RS        10
10           Londrina     PR        11
11             Cuiabá     MT        12
12           Brasília     DF        13
13             Maceió     AL        14
14        Nova Iguaçu     RJ        15
15            Goiânia     GO        16
16              Natal     RN        17
17          Boa Vista     RR        18
18             Palmas     TO        19
19       Porto Alegre     RS        20
20           Campinas     SP        21


In [29]:
geolocator = Nominatim(user_agent="geoapi_df_cidades")

# Função para buscar coordenadas com cidade e estado
def get_coordinates(cidade, estado):
    try:
        query = f"{cidade}, {estado}, Brasil"
        location = geolocator.geocode(query)
        if location:
            return location.latitude, location.longitude
        else:
            return None, None
    except:
        return None, None

latitudes = []
longitudes = []

for idx, row in df_cidades.iterrows():
    lat, lon = get_coordinates(row['Cidade'], row['Estado'])
    latitudes.append(lat)
    longitudes.append(lon)
    time.sleep(1)  # para evitar bloqueio da API

df_cidades['latitude'] = latitudes
df_cidades['longitude'] = longitudes

print(df_cidades)

               Cidade Estado  idCidade   latitude  longitude
0             Niterói     RJ         1 -22.888400 -43.114700
1           São Paulo     SP         2 -23.550651 -46.633382
2              Santos     SP         3 -23.933599 -46.328640
3    Feira de Santana     BA         4 -12.257893 -38.959805
4            Contagem     MG         5 -19.913275 -44.084095
5   Juazeiro do Norte     CE         6  -7.215345 -39.315334
6            São Luís     MA         7  -2.529526 -44.296394
7      Rio de Janeiro     RJ         8 -22.911014 -43.209373
8       Florianópolis     SC         9 -27.597300 -48.549610
9       Caxias do Sul     RS        10 -29.168505 -51.179639
10           Londrina     PR        11 -23.311288 -51.159502
11             Cuiabá     MT        12 -15.598669 -56.099130
12           Brasília     DF        13 -15.793987 -47.882800
13             Maceió     AL        14  -9.647684 -35.733926
14        Nova Iguaçu     RJ        15 -22.759217 -43.450873
15            Goiânia   

In [30]:
# Fazer um merge entre (colaborador e idCidade)
df_colaboradores = df_colaboradores.merge(
    df_cidades[['Cidade', 'Estado', 'idCidade']],
    on=['Cidade', 'Estado'],
    how='left'
)
print(df_colaboradores)

    ID              Nome                         Email             CPF  \
0    7  Vanessa Nogueira  vanessa.nogueira@exemplo.com  345.123.678-90   
1    8    Eduardo Araújo        edu.araújo@empresa.com  024.878.550-22   
2    9     Mariana Silva        mariana@dominio.com.br  123.456.789-00   
3   10      Rafael Costa        rafael.costa@email.com  789.654.321-00   
4   11      Thiago Moura            thiago@dominio.com  123.456.789-00   
..  ..               ...                           ...             ...   
75  76     Carla Martins        carla.martins@mail.com  456.123.987-12   
76  77      Roberto Dias        roberto.dias@email.com  789.456.789-12   
77  78     Larissa Costa        larissa.costa@mail.com  321.987.123-12   
78  79   Ricardo Almeida   ricardo.almeida@empresa.com  654.123.456-12   
79  80    Patrícia Souza    patricia.souza@dominio.com  789.321.789-12   

   Data de Nascimento             Telefone Estado            Cidade  \
0          1992-09-05   +55 (21) 9876-12

In [31]:
# Remover a coluna Estado (ela foi necessária apenas para aumentar precisão da geolocalização)
df_cidades = df_cidades.drop(columns=['Estado'])
print(df_cidades)

               Cidade  idCidade   latitude  longitude
0             Niterói         1 -22.888400 -43.114700
1           São Paulo         2 -23.550651 -46.633382
2              Santos         3 -23.933599 -46.328640
3    Feira de Santana         4 -12.257893 -38.959805
4            Contagem         5 -19.913275 -44.084095
5   Juazeiro do Norte         6  -7.215345 -39.315334
6            São Luís         7  -2.529526 -44.296394
7      Rio de Janeiro         8 -22.911014 -43.209373
8       Florianópolis         9 -27.597300 -48.549610
9       Caxias do Sul        10 -29.168505 -51.179639
10           Londrina        11 -23.311288 -51.159502
11             Cuiabá        12 -15.598669 -56.099130
12           Brasília        13 -15.793987 -47.882800
13             Maceió        14  -9.647684 -35.733926
14        Nova Iguaçu        15 -22.759217 -43.450873
15            Goiânia        16 -16.680882 -49.253269
16              Natal        17  -5.805398 -35.208090
17          Boa Vista       

In [32]:
# Colocar a coluna idCidade na primeira posição
df_cidades = df_cidades[['idCidade', 'Cidade', 'latitude', 'longitude']]
print(df_cidades)

    idCidade             Cidade   latitude  longitude
0          1            Niterói -22.888400 -43.114700
1          2          São Paulo -23.550651 -46.633382
2          3             Santos -23.933599 -46.328640
3          4   Feira de Santana -12.257893 -38.959805
4          5           Contagem -19.913275 -44.084095
5          6  Juazeiro do Norte  -7.215345 -39.315334
6          7           São Luís  -2.529526 -44.296394
7          8     Rio de Janeiro -22.911014 -43.209373
8          9      Florianópolis -27.597300 -48.549610
9         10      Caxias do Sul -29.168505 -51.179639
10        11           Londrina -23.311288 -51.159502
11        12             Cuiabá -15.598669 -56.099130
12        13           Brasília -15.793987 -47.882800
13        14             Maceió  -9.647684 -35.733926
14        15        Nova Iguaçu -22.759217 -43.450873
15        16            Goiânia -16.680882 -49.253269
16        17              Natal  -5.805398 -35.208090
17        18          Boa Vi

In [33]:
# Remover a coluna cidade do df_colaboradores (pois agora essa informação está no df_cidades)
df_colaboradores = df_colaboradores.drop(columns=['Cidade'])
display(df_colaboradores)

,ID,Nome,Email,CPF,Data de Nascimento,Telefone,Estado,Endereço,Telefone_status,Situacao_CPF,idCidade
0,7,Vanessa Nogueira,vanessa.nogueira@exemplo.com,345.123.678-90,1992-09-05,+55 (21) 9876-1234,RJ,"Rua Mato Grosso, 192, Niterói - RJ",Inválido: celular com tamanho incorreto,Válido,1
1,8,Eduardo Araújo,edu.araújo@empresa.com,024.878.550-22,1980-07-30,+55 (11) 93220-4567,SP,"Rua Rio Branco, 831, São Paulo - SP",Válido,Válido,2
2,9,Mariana Silva,mariana@dominio.com.br,123.456.789-00,1999-06-15,+55 (19) 99765-3332,SP,"Rua Horizonte Azul, 779, Santos - SP",Válido,Válido,3
3,10,Rafael Costa,rafael.costa@email.com,789.654.321-00,1987-03-22,+55 (71) 91234-5678,BA,"Rua Cruzeiro do Sul, 162, Feira de Santana - BA",Válido,Válido,4
4,11,Thiago Moura,thiago@dominio.com,123.456.789-00,1990-01-12,+55 (21) 99999-0001,RJ,"Rua do Sol, 121, Niterói - RJ",Válido,Válido,1
...,...,...,...,...,...,...,...,...,...,...,...
75,76,Carla Martins,carla.martins@mail.com,456.123.987-12,1992-05-05,+55 (41) 91234-5789,RJ,"Rua Bela Vista, 718, Nova Iguaçu - RJ",Válido,Válido,15
76,77,Roberto Dias,roberto.dias@email.com,789.456.789-12,1989-07-22,+55 (41) 99887-7776,RJ,"Rua Alvorada, 163, Nova Iguaçu - RJ",Válido,Válido,15
77,78,Larissa Costa,larissa.costa@mail.com,321.987.123-12,1993-03-15,+55 (41) 98765-4321,RJ,"Rua Bela Vista, 540, Nova Iguaçu - RJ",Válido,Válido,15
78,79,Ricardo Almeida,ricardo.almeida@empresa.com,654.123.456-12,1990-09-09,+55 (41) 91234-4333,RJ,"Rua Bela Vista, 859, Nova Iguaçu - RJ",Válido,Válido,15


In [34]:
# Agora tratar a coluna endereço (remover a cidade e estado, pois essas informações já estão aparecendo em outras colunas)
df_colaboradores['Endereço'] = df_colaboradores['Endereço'].str.extract(r'^(.+?,\s*\d+)')
print(df_colaboradores)

    ID              Nome                         Email             CPF  \
0    7  Vanessa Nogueira  vanessa.nogueira@exemplo.com  345.123.678-90   
1    8    Eduardo Araújo        edu.araújo@empresa.com  024.878.550-22   
2    9     Mariana Silva        mariana@dominio.com.br  123.456.789-00   
3   10      Rafael Costa        rafael.costa@email.com  789.654.321-00   
4   11      Thiago Moura            thiago@dominio.com  123.456.789-00   
..  ..               ...                           ...             ...   
75  76     Carla Martins        carla.martins@mail.com  456.123.987-12   
76  77      Roberto Dias        roberto.dias@email.com  789.456.789-12   
77  78     Larissa Costa        larissa.costa@mail.com  321.987.123-12   
78  79   Ricardo Almeida   ricardo.almeida@empresa.com  654.123.456-12   
79  80    Patrícia Souza    patricia.souza@dominio.com  789.321.789-12   

   Data de Nascimento             Telefone Estado                  Endereço  \
0          1992-09-05   +55 (21)

In [35]:
df_colaboradores.head(10)

,ID,Nome,Email,CPF,Data de Nascimento,Telefone,Estado,Endereço,Telefone_status,Situacao_CPF,idCidade
0,7,Vanessa Nogueira,vanessa.nogueira@exemplo.com,345.123.678-90,1992-09-05,+55 (21) 9876-1234,RJ,"Rua Mato Grosso, 192",Inválido: celular com tamanho incorreto,Válido,1
1,8,Eduardo Araújo,edu.araújo@empresa.com,024.878.550-22,1980-07-30,+55 (11) 93220-4567,SP,"Rua Rio Branco, 831",Válido,Válido,2
2,9,Mariana Silva,mariana@dominio.com.br,123.456.789-00,1999-06-15,+55 (19) 99765-3332,SP,"Rua Horizonte Azul, 779",Válido,Válido,3
3,10,Rafael Costa,rafael.costa@email.com,789.654.321-00,1987-03-22,+55 (71) 91234-5678,BA,"Rua Cruzeiro do Sul, 162",Válido,Válido,4
4,11,Thiago Moura,thiago@dominio.com,123.456.789-00,1990-01-12,+55 (21) 99999-0001,RJ,"Rua do Sol, 121",Válido,Válido,1
5,12,Camila Andrade,camila.andrade@dominio.com,987.654.321-00,1993-03-01,+55 (11) 91234-5678,SP,"Rua Esperança, 999",Válido,Válido,3
6,13,Bruno Oliveira,bruno@empresa.com,000.000.000-00,1985-07-30,+55 (31) 9999-8888,MG,"Rua Bela Vista, 648",Inválido: celular com tamanho incorreto,Inválido: todos os dígitos são iguais,5
7,14,Ana Carolina,ana.carol@email.com,321.654.987-00,1994-02-14,+55 (71) 98877-6655,BA,"Rua Acre, 555",Válido,Válido,4
8,15,Fernando Reis,fernando.reis@empresa.com.br,789.654.123-00,1987-09-05,+55 (85) 92345-6789,CE,"Rua Bela Vista, 658",Válido,Válido,6
9,16,Isabela Monteiro,isabela.monteiro@gmail.com,741.852.963-00,1982-10-31,+55 (98) 99876-5432,MA,"Rua Dom Pedro I, 997",Válido,Válido,7


In [ ]:
# Salvar os dataframes em .csv
# df_colaboradores.to_csv('../ArquivosExternos/dados_tratados_v2/dc_DadosPrimarios_tratado.csv', index=False, encoding='utf-8')
# df_cidades.to_csv('../ArquivosExternos/dados_tratados_v2/dim_Cidades.csv', index=False, encoding='utf-8')

In [ ]:

# variável de ambiente para construir o caminho
caminho_base_arquivos = os.environ.get("CAMINHO_ARQUIVOS_CSV")
caminho_saida = os.path.join(caminho_base_arquivos, "dados_tratados_v2", "dc_DadosPrimarios_tratado.csv")

# variável caminho_saida
df_colaboradores.to_csv(caminho_saida, index=False)

In [ ]:

# variável de ambiente para construir o caminho
caminho_base_arquivos = os.environ.get("CAMINHO_ARQUIVOS_CSV")
caminho_saida = os.path.join(caminho_base_arquivos, "dados_tratados_v2", "dim_Cidades.csv")

# variável caminho_saida
df_cidades.to_csv(caminho_saida, index=False)